In [244]:
import numpy as np
import scipy as sp
import scipy.optimize as so
import matplotlib.pyplot as plt
%matplotlib inline

In [245]:
# solve min 1/2 x^T A x + b^T x
# subject to x>=0

def LCP_PGD(A,b,tau,Nmax,x0):
    xkm1=np.zeros(np.size(x0))
    ykm1=xkm1 # all 0
    
    # first step
    xk=xkm1 - 0.001*(A.dot(xkm1)+b)
    yk=A.dot(xk)
    ite=0
    for i in range(1,Nmax):
        ite+=1
        ykmykm1=yk-ykm1
        alphak=(xk-xkm1).dot(ykmykm1)/(ykmykm1).dot(ykmykm1) # Barzilai-Borwein
        xkp1=xk-alphak*(yk+b) # gradient descent
        xkp1=xkp1.clip(min=0)     # projection to feasible set x>=0
        resx=xkp1-xk
        ykp1=A.dot(xkp1)
        resb=ykp1+b
        if np.max(np.abs(resx)) < tau and np.min(resb) > - tau:
            break
        xkm1=xk
        ykm1=yk
        xk=xkp1
        yk=ykp1
    return xkp1,resid,ite


In [246]:
matDim=5
Aroot=np.random.randn(matDim,matDim)
Amat=np.transpose(Aroot).dot(Aroot)
bvec=np.random.randn(matDim)
print(Amat,bvec)

[[  1.89749107   4.06600865  -2.2006939    0.88533818   0.59490433]
 [  4.06600865  12.03983858  -1.23867513   0.37487382   2.38548815]
 [ -2.2006939   -1.23867513   9.9143505   -5.77625201   1.72271177]
 [  0.88533818   0.37487382  -5.77625201   6.581555    -3.22826802]
 [  0.59490433   2.38548815   1.72271177  -3.22826802   2.82037326]] [-0.36002932 -0.11801724  1.08035237 -1.20715816  1.06953444]


In [247]:
restarget=1e-5

xsol,resid,ite=LCP_PGD(Amat,bvec,restarget,10000,np.zeros(np.size(bvec)))
print(xsol,resid,ite)

[ 0.14079874  0.          0.03705908  0.19699948  0.        ] [ -4.40353338e-06   3.10106581e-06  -1.19368660e-06   1.34873564e-06
  -2.94203875e-06] 76


In [248]:
y=Amat.dot(xsol)+bvec
print(y)
print(xsol)
for i in range(len(xsol)):
    if xsol[i]<- restarget:
        print("res.x error: ", xsol[i])
    if y[i]< - restarget:
        print("y error: ", y[i])
    print(y[i]*xsol[i])

[ -9.47911775e-06   4.82417463e-01  -4.54932001e-06  -3.29221430e-06
   5.81171206e-01]
[ 0.14079874  0.          0.03705908  0.19699948  0.        ]
-1.33464787272e-06
0.0
-1.68593592085e-07
-6.48564507078e-07
0.0


In [249]:
bnds = [(0,None) for i in range(matDim)]
fun= lambda x: 0.5*x.dot(Amat.dot(x))+bvec.dot(x)
print(bnds)
res=so.minimize(fun, np.zeros(np.size(bvec)), method='L-BFGS-B', bounds=bnds,tol=restarget)
print(res)

[(0, None), (0, None), (0, None), (0, None), (0, None)]
      fun: -0.12423320137888973
 hess_inv: <5x5 LbfgsInvHessProduct with dtype=float64>
      jac: array([  8.80712170e-05,   4.83039857e-01,   6.99405811e-04,
        -3.66669195e-04,   5.81395358e-01])
  message: b'CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH'
     nfev: 54
      nit: 7
   status: 0
  success: True
        x: array([ 0.14098958,  0.        ,  0.03719461,  0.19703754,  0.        ])


In [250]:
y=Amat.dot(res.x)+bvec
print(y)
print(res.x)
for i in range(len(res.x)):
    if res.x[i]<-1e-5:
        print("res.x error")
    if y[i]<-1e-5:
        print("y error")
    print(y[i]*res.x[i])

[  8.80614364e-05   4.83039797e-01   6.99355017e-04  -3.66704030e-04
   5.81395345e-01]
[ 0.14098958  0.          0.03719461  0.19703754  0.        ]
1.24157451058e-05
0.0
2.60122375764e-05
y error
-7.2254461642e-05
0.0


In [251]:
print(res.x)
print(xsol)
print(res.x-xsol)

[ 0.14098958  0.          0.03719461  0.19703754  0.        ]
[ 0.14079874  0.          0.03705908  0.19699948  0.        ]
[  1.90837998e-04   0.00000000e+00   1.35535543e-04   3.80638722e-05
   0.00000000e+00]


In [252]:
def GD(A,b,tau,Nmax,x0):
    xkm1=np.zeros(np.size(x0))
    ykm1=xkm1 # all 0
    
    # first step
    xk=xkm1 - 0.001*(A.dot(xkm1)+b)
    ite=0
    for i in range(1,Nmax):
        ite+=1
        yk=A.dot(xk)
        ykmykm1=yk-ykm1
        alphak=(xk-xkm1).dot(ykmykm1)/(ykmykm1).dot(ykmykm1) # Barzilai-Borwein
        xkp1=xk-alphak*(yk+b) # gradient descent
       # xkp1=xkp1.clip(min=0)     # projection to feasible set x>=0
        resid=xkp1-xk
        if np.max(np.abs(resid)) < tau:
            break
        xkm1=xk
        ykm1=yk
        xk=xkp1
    return xkp1,resid,ite

In [253]:
xsol,resid,ite=GD(Amat,bvec,0.00001,10000,np.zeros(np.size(bvec)))
print(xsol,resid,ite)

[ 0.79142284 -0.06891054  0.01095901 -0.34678718 -0.89150199] [ -1.64527425e-07   1.26580093e-07   9.70131144e-07  -2.89095518e-07
   1.08491837e-08] 48


In [254]:
res=so.minimize(fun, np.zeros(np.size(bvec)), method='L-BFGS-B',tol=restarget)
print(res)

      fun: -0.3999148797717671
 hess_inv: <5x5 LbfgsInvHessProduct with dtype=float64>
      jac: array([ -2.22044605e-07,  -2.37587727e-06,  -1.69309011e-06,
        -9.76996262e-07,   2.77555756e-07])
  message: b'CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL'
     nfev: 78
      nit: 10
   status: 0
  success: True
        x: array([ 0.79142311, -0.06891083,  0.01095739, -0.34678791, -0.89150257])
